In [10]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)

indian_locations = [
    "Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai",
    "Pune", "Kolkata", "Ahmedabad", "Jaipur", "Lucknow",
    "Visakhapatnam", "Bhopal", "Chandigarh", "Coimbatore", "Indore"
]

records = 10000
data = []

for _ in range(records):
    investment = random.randint(100000, 1000000)
    location = random.choice(indian_locations)
    competition = random.randint(1, 10)
    customers = random.randint(50, 1000)
    rent = random.randint(5000, 50000)
    marketing = random.randint(2000, 50000)

    # Business Logic Score
    score = (
        (investment / 100000) +
        customers/100 -
        competition*1.5 +
        marketing/10000 -
        rent/20000
    )

    # 3-Class Outcome Logic
    if score < 5:
        outcome = "Loss"
    elif 5 <= score < 9:
        outcome = "Medium"
    else:
        outcome = "Profitable"

    data.append([investment, location, competition,
                 customers, rent, marketing, outcome])

df = pd.DataFrame(data, columns=[
    "Investment", "Location", "CompetitionLevel",
    "TargetCustomers", "MonthlyRent",
    "MarketingBudget", "BusinessOutcome"
])

df.to_csv("business_data_multiclass.csv", index=False)
print("Dataset Generated Successfully")

Dataset Generated Successfully


In [11]:
import pandas as pd

df = pd.read_csv("business_data_multiclass.csv")

print(df.head())
print(df["BusinessOutcome"].value_counts())
print(df.info())

   Investment   Location  CompetitionLevel  TargetCustomers  MonthlyRent  \
0      371781      Delhi                 1              285        18044   
1      810369       Pune                 2              344        15713   
2      122217    Kolkata                 8              700        38580   
3      930970       Pune                 1              806        28147   
4      191330  Bangalore                 3               87        46296   

   MarketingBudget BusinessOutcome  
0             4085            Loss  
1            49899      Profitable  
2            24321            Loss  
3            43808      Profitable  
4            12504            Loss  
BusinessOutcome
Loss          5857
Medium        2138
Profitable    2005
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Investment        

In [12]:
df.isnull().sum()

Investment          0
Location            0
CompetitionLevel    0
TargetCustomers     0
MonthlyRent         0
MarketingBudget     0
BusinessOutcome     0
dtype: int64

In [13]:
print(df["BusinessOutcome"].unique())

['Loss' 'Profitable' 'Medium']


In [14]:
print(df["BusinessOutcome"].value_counts())

BusinessOutcome
Loss          5857
Medium        2138
Profitable    2005
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
categorical_cols = ['Location','BusinessOutcome']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = df.drop("BusinessOutcome", axis=1)
y = df["BusinessOutcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.939

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97      1186
           1       0.87      0.84      0.86       435
           2       0.94      0.92      0.93       379

    accuracy                           0.94      2000
   macro avg       0.92      0.91      0.92      2000
weighted avg       0.94      0.94      0.94      2000



In [19]:
probs = model.predict_proba(X_test)

print("Loss Probability:", probs[0][0] * 100)
print("Medium Probability:", probs[0][1] * 100)
print("Profit Probability:", probs[0][2] * 100)

Loss Probability: 16.0
Medium Probability: 64.0
Profit Probability: 20.0


In [20]:
df

,Investment,Location,CompetitionLevel,TargetCustomers,MonthlyRent,MarketingBudget,BusinessOutcome
0,371781,6,1,285,18044,4085,0
1,810369,13,2,344,15713,49899,2
2,122217,10,8,700,38580,24321,0
3,930970,13,1,806,28147,43808,2
4,191330,1,3,87,46296,12504,0
...,...,...,...,...,...,...,...
9995,477414,7,10,462,13066,45705,0
9996,114418,9,10,714,33046,45429,0
9997,200284,12,6,77,31974,14617,0
9998,925258,10,7,374,45047,39599,0


In [21]:
import pickle
with open("model.pkl","wb") as f:
    pickle.dump(model,f)

In [22]:
with open("label_encoders.pkl","wb") as f:
    pickle.dump(label_encoders,f)

In [23]:
features_order=list(X_train.columns)
with open("features_order.pkl","wb") as f:
    pickle.dump(features_order,f)